In [ ]:
%pip install astro-datalab
%pip install python-dotenv


# Imports necessários

In [1]:
import pandas as pd
import time
import pandas as pd
import requests
import os
from io import StringIO

## Baixa os fits G,R,I,Z e o catalogo dos campos presentes no array bricks, salvando no repositório /data/raw/images/dr10/south.

In [ ]:

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

bricks = ["0030m292","0030m295","0030m297","0030m300","0030m302","0030m307","0030m310","0030m312","0030m380", "0030m390","0001m007","0001m010","0001m012","0001m015","0001m017","0001m022",
         "0001m035","0001m042","0001m047","0001m052","0010m350","0010m352","0010m367","0011m412","0011p002","0011p005","0011p017","0011p027","0011p045","0011p047",
         "0011p050","0020m575","0021m005","0021m007","0021m010","0021m017","0021m020","0021m040","0021m402","0021m510","0021m585","0021p005","0040m397","0040m400"
         ,"0040m402","0040m445","0040m495","0040m540","0040m585","0040m622","0041m020","0041m042","0041m065","0041m087","0041m090","0041m462",
         "0041m470","0041m505","0041m512","0041m592","0041m595","0041p010","0041p012","0041p030","0042m325","0042m377","0042m422","0042m432","0042m472",
         "0042m605","0043m042","0043m050","0043m067","0043m342","0043m387","0043m392","0043m445","0043m487","0043m490","0043m567","0043m570","0043m575","0050m140","0050m142","0050m145","0050m300",
         "0050m302","0050m347","0050m355","0050m357","0050m402","0050m515","0050m552","0051m047","0051m062","0051m097","0051m320","0051m450","0051m487"
         ,"0051m495","0051m625","0051p037","0051p045","0052m470","0053m012","0053m017","0053m035","0060m125","0060m440","0060m530","0060m592","0030m297"]
for i in range(len(bricks)):  # baixa os dados de cada campo
    brick_name = bricks[i]
    prefix = brick_name[:3]
    # URL oDR10 South
    base_url = f"https://portal.nersc.gov/cfs/cosmo/data/legacysurvey/dr10/south/coadd/{prefix}/{brick_name}/"
    files_to_download = {
      "banda_g": f"legacysurvey-{brick_name}-image-g.fits.fz",
      "banda_r": f"legacysurvey-{brick_name}-image-r.fits.fz",
      "banda_i": f"legacysurvey-{brick_name}-image-i.fits.fz",
       "banda_z": f"legacysurvey-{brick_name}-image-z.fits.fz",
       "catalogo": f"tractor-{brick_name}.fits"
    }
    save_path = f"./data/raw/images/dr10/south/{prefix}/{brick_name}/" # arquivo onde vai falvar os fitz
    os.makedirs(save_path, exist_ok=True) 
    print(f"\n--- Iniciando Brick {i+1}: {brick_name} ---")

    for key, filename in files_to_download.items():
            if key == "catalogo":
                url = f"https://portal.nersc.gov/cfs/cosmo/data/legacysurvey/dr10/south/tractor/{prefix}/"+filename
            else:
                url = base_url + filename
            path = os.path.join(save_path, filename)
            if os.path.exists(path) and os.path.getsize(path) > 1024:#  pula o download se já existir o arquvio e não estiver vazio ( > 1KB)
                print(f"{filename} já existe e parece íntegro. Pulando...")
                continue
            print(f"Baixando {key}...")
            try:
                with requests.get(url, stream=True, timeout=60, headers=headers) as r: # Adicionando Headers e Timeout
                    if r.status_code == 200:
                        with open(path, 'wb') as f:
                            for chunk in r.iter_content(chunk_size=1024*1024): # Chunk de 1MB é mais rápido
                                if chunk:
                                    f.write(chunk)
                        if os.path.getsize(path) < 1024: # checagem se baixou o conteúdo do arquivo ou deu algum tipo de erro
                            print(f"!!! ERRO: {filename} baixou vazio. Removendo...")
                            os.remove(path)
                        else:
                            print(f" OK: {filename} salvo ({os.path.getsize(path)/1024/1024:.2f} MB)")
                    else:
                        print(f" Erro {r.status_code} no brick {brick_name}.")
            except Exception as e:
                print(f" Erro na conexão: {e}")
                # Se deu erro no meio do download, remove o arquivo quebrado
                if os.path.exists(path):
                    os.remove(path)

print("bricks prontos")


In [3]:
print(len(bricks))

112
